# Part B: 07 — Machine Learning Models\n\nGenerates 1:2 R:R triple-barrier target labels and trains classification models (Random Forest, HistGradientBoosting, Logistic Regression).

In [1]:
import sys, os
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

from label_generator import LabelGenerator
from validation_utils import ValidationUtils
from model_evaluator import ModelEvaluator

df = pd.read_parquet('../data/features/xauusd_features.parquet')
df_labeled = LabelGenerator.triple_barrier_labels(df, risk_rr=2.0, horizon=15)

train_df, val_df, test_df = ValidationUtils.chronological_split(df_labeled, 0.6, 0.2)
feature_cols = [c for c in df.select_dtypes(include=['float64', 'int64', 'bool']).columns if c not in ['target_label', 'mfe', 'mae']]

X_train, y_train = train_df[feature_cols], train_df['target_label']
X_test, y_test = test_df[feature_cols], test_df['target_label']

clf = HistGradientBoostingClassifier(random_state=42)
clf.fit(X_train, y_train)
preds = clf.predict(X_test)
metrics = ModelEvaluator.evaluate_classification(y_test, preds)
print("HistGradientBoosting test performance:", metrics)


HistGradientBoosting test performance: {'accuracy': 0.24293785310734464, 'precision': 0.6076305496466068, 'recall': 0.24293785310734464, 'f1': 0.28225573950710975}
